<a href="https://colab.research.google.com/github/Samridh29-8-2005/TIC-TAC-TOE-AI-/blob/main/TIC_TAC_TOE_AI_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================
# PART 1: BACKEND - Game Logic & Rules
# ==========================================

class TicTacToe:
    def __init__(self):
        self.board = [' ' for _ in range(9)] # 3x3 board flattened into 1D list
        self.current_winner = None

    def available_moves(self):
        # Returns indices of empty squares
        return [i for i, spot in enumerate(self.board) if spot == ' ']

    def empty_squares(self):
        # Checks if any moves are left
        return ' ' in self.board

    def make_move(self, square, letter):
        # Places a letter (X or O) on the board
        if self.board[square] == ' ':
            self.board[square] = letter
            if self.check_winner(square, letter):
                self.current_winner = letter
            return True
        return False

    def undo_move(self, square):
        # Removes a letter (Used by AI to simulate moves)
        self.board[square] = ' '
        self.current_winner = None

    def check_winner(self, square, letter):
        # Checks if the last move resulted in a win
        # Check row
        row_ind = square // 3
        row = self.board[row_ind*3:(row_ind+1)*3]
        if all([spot == letter for spot in row]):
            return True
        # Check column
        col_ind = square % 3
        column = [self.board[row_ind*3 + col_ind] for row_ind in range(3)]
        if all([spot == letter for spot in column]):
            return True
        # Check diagonals
        if square % 2 == 0:
            diagonal1 = [self.board[i] for i in [0, 4, 8]]
            if all([spot == letter for spot in diagonal1]):
                return True
            diagonal2 = [self.board[i] for i in [2, 4, 6]]
            if all([spot == letter for spot in diagonal2]):
                return True
        return False

    def evaluate(self, ai_letter, human_letter):
        # Gives a score to the current board state
        if self.current_winner == ai_letter:
            return 1   # AI Wins
        elif self.current_winner == human_letter:
            return -1  # Human Wins
        else:
            return 0   # Draw or Ongoing

In [2]:
# ==========================================
# PART 2: BACKEND - AI Algorithm (Minimax + Alpha-Beta)
# ==========================================

def minimax(board, depth, is_maximizing, alpha, beta, ai_letter, human_letter):
    # Step 1: Evaluate the current board state (Base Cases)
    score = board.evaluate(ai_letter, human_letter)
    if score == 1:   # AI wins scenario
        return score
    if score == -1:  # Human wins scenario
        return score
    if not board.empty_squares(): # Tie scenario
        return 0

    # Step 2: AI's turn (Maximizing)
    if is_maximizing:
        max_eval = -float('inf')
        for move in board.available_moves():
            board.make_move(move, ai_letter) # Try the move
            eval = minimax(board, depth + 1, False, alpha, beta, ai_letter, human_letter)
            board.undo_move(move)            # Undo the move

            max_eval = max(max_eval, eval)
            alpha = max(alpha, eval)
            if beta <= alpha: # Alpha-Beta Pruning (Stop searching this branch)
                break
        return max_eval

    # Step 3: Human's turn (Minimizing)
    else:
        min_eval = float('inf')
        for move in board.available_moves():
            board.make_move(move, human_letter) # Try the move
            eval = minimax(board, depth + 1, True, alpha, beta, ai_letter, human_letter)
            board.undo_move(move)               # Undo the move

            min_eval = min(min_eval, eval)
            beta = min(beta, eval)
            if beta <= alpha: # Alpha-Beta Pruning (Stop searching this branch)
                break
        return min_eval

def get_best_move(board, ai_letter, human_letter):
    # This function finds the absolute best move for the AI
    best_score = -float('inf')
    best_move = None

    for move in board.available_moves():
        board.make_move(move, ai_letter)
        score = minimax(board, 0, False, -float('inf'), float('inf'), ai_letter, human_letter)
        board.undo_move(move)

        if score > best_score:
            best_score = score
            best_move = move

    return best_move

In [3]:
# ==========================================
# PART 3: FRONTEND - Interactive UI for Colab
# ==========================================

import ipywidgets as widgets
from IPython.display import display, clear_output

class TicTacToeUI:
    def __init__(self):
        # Initialize game variables
        self.game = TicTacToe() # From Part 1
        self.human = 'X'
        self.ai = 'O'           # Uses Part 2
        self.game_over = False

        # Create 9 interactive buttons
        self.buttons = [widgets.Button(description=' ',
                                       layout=widgets.Layout(width='60px', height='60px'),
                                       style=widgets.ButtonStyle(button_color='lightgray', font_size='20px', font_weight='bold'))
                        for _ in range(9)]

        # Status display and Restart button
        self.status_label = widgets.Label(value="Your turn (X)", layout=widgets.Layout(margin='10px 0px'))
        self.reset_button = widgets.Button(description="Restart Game", button_style='info')

        # Arrange buttons in a 3x3 Grid
        self.grid = widgets.GridspecLayout(3, 3, grid_gap='5px')
        for i in range(3):
            for j in range(3):
                index = i * 3 + j
                self.grid[i, j] = self.buttons[index]
                self.buttons[index].on_click(self.on_button_click)

        self.reset_button.on_click(self.reset_game)

        # Render the complete UI
        self.ui = widgets.VBox([
            widgets.HTML(value="<h1 style='text-align:center;'>Tic-Tac-Toe AI</h1>"),
            self.status_label,
            self.grid,
            self.reset_button
        ], layout=widgets.Layout(align_items='center'))

    def on_button_click(self, btn):
        # Ignore clicks if game is over or square is taken
        if self.game_over or btn.description != ' ':
            return

        # --- HUMAN MOVE ---
        index = self.buttons.index(btn)
        self.game.make_move(index, self.human)
        btn.description = self.human
        btn.style.button_color = 'lightblue'

        if self.check_game_end():
            return

        # --- AI MOVE ---
        self.status_label.value = "AI is thinking..."
        best_move = get_best_move(self.game, self.ai, self.human) # Call AI Function

        if best_move is not None:
            self.game.make_move(best_move, self.ai)
            ai_btn = self.buttons[best_move]
            ai_btn.description = self.ai
            ai_btn.style.button_color = 'lightcoral'

        if not self.check_game_end():
            self.status_label.value = "Your turn (X)"

    def check_game_end(self):
        # Check who won or if it's a tie
        if self.game.current_winner == self.human:
            self.status_label.value = "You Win! (Impossible 😅)"
            self.game_over = True
            return True
        elif self.game.current_winner == self.ai:
            self.status_label.value = "AI Wins! 🤖"
            self.game_over = True
            return True
        elif not self.game.empty_squares():
            self.status_label.value = "It's a Tie! 🤝"
            self.game_over = True
            return True
        return False

    def reset_game(self, btn):
        # Reset board and UI to start fresh
        self.game = TicTacToe()
        self.game_over = False
        self.status_label.value = "Your turn (X)"
        for button in self.buttons:
            button.description = ' '
            button.style.button_color = 'lightgray'

# ==========================================
# EXECUTION: Run this to start the game
# ==========================================
game_ui = TicTacToeUI()
display(game_ui.ui)